In [68]:
"""
investigate correlation in the following pairs, currently we have all predictions (x1, x2, x3, ablation, aggregate)

(1) x1+x2 vs x3
(2) x1+x3 vs x2
(3) x2+x3 vs x1

"""

'\ninvestigate correlation in the following pairs, currently we have all predictions (x1, x2, x3, ablation, aggregate)\n\n(1) x1+x2 vs x3\n(2) x1+x3 vs x2\n(3) x2+x3 vs x1\n\n'

In [1]:
import json
with open('JAADbeh_results.json', 'r') as f:
    results = json.load(f)

In [2]:
results.keys()

dict_keys(['x1', 'x2', 'x3', 'x1_x2', 'x1_x3', 'x2_x3', 'x1_x2_x3'])

In [9]:
import numpy as np
len(results['x1']['test_gts']), np.sum(np.array(results['x1']['test_gts']).squeeze())

(78870, 32095.0)

In [10]:
78870 - 32095

46775

In [19]:
78870/5

15774.0

In [17]:
def count_imbalance(results):
    total = len(results['x1']['test_gts'])
    pos = np.sum(np.array(results['x1']['test_gts']).squeeze())
    neg = total- pos
    norm_pos = pos/total
    norm_neg = neg/total
    print('pos:neg = {0:.2f}:{1:.2f}'.format(norm_pos, norm_neg))
    

In [18]:
count_imbalance(results)

pos:neg = 0.41:0.59


In [2]:
def count_correlated_mistakes(preds_1, preds_2, gt):
    agreement = preds_1==preds_2
    stack_agreement_preds = []
    stack_gts = []

    for i, a in enumerate(agreement):
        if a==True:#two models agree each other
            stack_agreement_preds.append(int(preds_1[i]))
            stack_gts.append(int(gt[i]))
    aggre_num = len(np.array(stack_agreement_preds))
    mistakes_num = sum(np.array(stack_agreement_preds)!=np.array(stack_gts))
    
    return aggre_num, mistakes_num

def count_mistake_union(preds_1, preds_2, gt):
    mistake_1 = []#agreement
    mistake_2 = []#agreed mistakes
    
    for i, v in enumerate(gt):
        if int(gt[i])!= int(preds_1[i]):
            mistake_1.append(i)
        
        if int(gt[i])!= int(preds_2[i]):
            mistake_2.append(i)
    return len(set(list(set(mistake_1)) + list(set(mistake_2))))


def count_indivisual_mistakes(preds, gt):
    return sum(preds!=gt)

def get_conf_coef(probs_1, probs_2):
    return np.corrcoef(np.squeeze(probs_1), np.squeeze(probs_2))#x2_x3

# x1

In [19]:
def flatten_transformNP(model_name):
    preds = np.array(results[model_name]['preds'])
    src_li = np.array(results[model_name]['test_gts'])
    if not preds.shape == src_li.shape:
        mod_li = list(itertools.chain(*src_li))
        print(count_indivisual_mistakes(preds, mod_li))
    else:
        print(count_indivisual_mistakes(preds,src_li))

In [50]:
flatten_transformNP(model_name='x1')

33008


# x2

In [51]:
flatten_transformNP(model_name='x2')

31315


# x3

In [52]:
flatten_transformNP(model_name='x3')

30448


# x1 + x2

In [53]:
base = 'x1'
add = 'x2'
preds_1 = np.array(results[base]['preds'])
preds_2 = np.array(results[add]['preds'])
gt = np.array(results[base]['test_gts'])
gt = list(itertools.chain(*gt))
print('agreed prediction and agreed mistakes')
print(count_correlated_mistakes(preds_1, preds_2, gt))
print('actual mistakes')
merged = base+'_'+add
preds = np.array(results[merged]['preds'])
print(count_indivisual_mistakes(preds, gt))
print('mistake union')
print(count_mistake_union(preds_1, preds_2, gt))
print('correlation coefficient')
probs_1 = np.array(results[base]['probs'])
probs_2 = np.array(results[add]['probs'])
print(get_conf_coef(probs_1, probs_2))
print('total gt')
print(len(gt))

agreed prediction and agreed mistakes
(44073, 14763)
actual mistakes
32722
mistake union
49560
correlation coefficient
[[1.         0.21328769]
 [0.21328769 1.        ]]
total gt
78870


# x1+x3

In [62]:
base = 'x1'
add = 'x3'
def get_all_score(base, add):
    preds_1 = np.array(results[base]['preds'])
    preds_2 = np.array(results[add]['preds'])
    gt = np.array(results[base]['test_gts'])
    if not gt.shape == preds_2.shape:
        gt = list(itertools.chain(*gt))
    print('agreed prediction and agreed mistakes')
    print(count_correlated_mistakes(preds_1, preds_2, gt))
    print('actual mistakes')
    merged = base+'_'+add
    if len(merged.split('_'))==3:
        merged = 'x1_x2_x3'
    preds = np.array(results[merged]['preds'])
    print(count_indivisual_mistakes(preds, gt))
    print('mistake union')
    print(count_mistake_union(preds_1, preds_2, gt))
    print('correlation coefficient')
    probs_1 = np.array(results[base]['probs'])
    probs_2 = np.array(results[add]['probs'])
    print(get_conf_coef(probs_1, probs_2))
    print('base model mistakes')
    print(count_indivisual_mistakes(preds_1, gt))
    print('additional model mistakes')
    print(count_indivisual_mistakes(preds_2, gt))
    print('total gt')
    print(len(gt))

In [55]:
get_all_score(base, add)

agreed prediction and agreed mistakes
(49602, 17094)
actual mistakes
31761
mistake union
46362
correlation coefficient
[[1.         0.23828012]
 [0.23828012 1.        ]]
total gt
78870


# x2 + x3

In [56]:
base = 'x2'
add = 'x3'
get_all_score(base, add)

agreed prediction and agreed mistakes
(49637, 16265)
actual mistakes
27314
mistake union
45498
correlation coefficient
[[1.         0.41274306]
 [0.41274306 1.        ]]
total gt
78870


# (1) x1+x2 vs x3

In [64]:
base = 'x1_x2'
add = 'x3'
get_all_score(base, add)

agreed prediction and agreed mistakes
(50484, 17392)
actual mistakes
31668
mistake union
45778
correlation coefficient
[[1.         0.26950737]
 [0.26950737 1.        ]]
base model mistakes
32722
additional model mistakes
30448
total gt
78870


# (2) x1+x3 vs x2

In [63]:
base = 'x1_x3'
add = 'x2'
get_all_score(base, add)

agreed prediction and agreed mistakes
(46110, 15158)
actual mistakes
31668
mistake union
47918
correlation coefficient
[[1.        0.3635969]
 [0.3635969 1.       ]]
base model mistakes
31761
additional model mistakes
31315
total gt
78870


# (3) x2+x3 vs x1

In [65]:
base = 'x2_x3'
add = 'x1'
get_all_score(base, add)

agreed prediction and agreed mistakes
(51690, 16571)
actual mistakes
31668
mistake union
43751
correlation coefficient
[[1.         0.27552503]
 [0.27552503 1.        ]]
base model mistakes
27314
additional model mistakes
33008
total gt
78870


# x1_x2_x3 mistakes

In [66]:
x1_x2_x3_preds = np.array(results['x1_x2_x3']['preds'])
count_indivisual_mistakes(x1_x2_x3_preds, gt)

31668